# EFGP-SING vs SING-SparseGP: Three Benchmark Experiments

Three 2-D latent SDE settings with increasing drift nonlinearity. Each row shows:
- **Drift recovery**: true / EFGP / SparseGP quiver plots with RMSE
- **Hyperparameter paths**: ℓ and σ² over EM iterations for EFGP; SparseGP final value marked
- **Summary table**

A final scatter plot shows relative drift RMSE vs wall time across all settings.


In [ ]:
import sys
from pathlib import Path
_SING = Path('/Users/colecitrenbaum/Documents/GPs/sing')
if str(_SING) not in sys.path:
    sys.path.insert(0, str(_SING))

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgs

from sing.likelihoods import Likelihood
from sing.simulate_data import simulate_sde, simulate_gaussian_obs
from sing.sde import SparseGP
from sing.kernels import RBF
from sing.expectation import GaussHermiteQuadrature
from sing.sing import fit_variational_em
from sing.initialization import initialize_zs
import sing.efgp_em as em
import sing.efgp_jax_primitives as jp
import sing.efgp_jax_drift as jpd

print('jax devices:', jax.devices())


In [ ]:
class _GLik(Likelihood):
    """Minimal Gaussian likelihood for EFGP (EFGP manages emissions separately)."""
    def ell(self, y, mean, var, output_params):
        R = output_params['R']
        return jnp.sum(-0.5 * jnp.log(2 * jnp.pi * R)
                       - 0.5 * ((y - mean)**2 + var) / R)

def _make_data(drift_fn, x0, T, t_max, sigma, N_obs, seed, D=2):
    xs = simulate_sde(jr.PRNGKey(seed), x0=x0, f=drift_fn,
                      t_max=t_max, n_timesteps=T,
                      sigma=lambda x, t: sigma * jnp.eye(D))
    rng = np.random.default_rng(seed)
    C_true = rng.standard_normal((N_obs, D)) * 0.5
    out_true = dict(C=jnp.asarray(C_true), d=jnp.zeros(N_obs),
                    R=jnp.full((N_obs,), 0.05))
    ys = simulate_gaussian_obs(jr.PRNGKey(seed + 1), xs, out_true)
    return np.asarray(xs), ys

def _shared_init(ys, D, N_obs, t_max, T):
    yc = ys - ys.mean(0)
    _, _, Vt = jnp.linalg.svd(yc, full_matrices=False)
    op = dict(C=Vt[:D].T, d=ys.mean(0), R=jnp.full((N_obs,), 0.1))
    ip = jax.tree_util.tree_map(
        lambda x: x[None], dict(mu0=jnp.zeros(D), V0=jnp.eye(D) * 0.1))
    t_grid = jnp.linspace(0., t_max, T)
    return op, ip, t_grid

def procrustes_align(m_inf, m_true):
    Xi, Xt = np.asarray(m_inf), np.asarray(m_true)
    bi, bt = Xi.mean(0), Xt.mean(0)
    A_T, *_ = np.linalg.lstsq(Xi - bi, Xt - bt, rcond=None)
    A = A_T.T
    return A, bt - A @ bi          # A, b  s.t.  Xi @ A.T + b \u2248 Xt

def run_efgp(lik, t_grid, op_init, ip_init, D, sigma, n_em, n_estep,
             rho_sched, n_mstep, mstep_lr):
    t0 = time.time()
    mp, _, op, _, hist = em.fit_efgp_sing_jax(
        likelihood=lik, t_grid=t_grid,
        output_params=op_init, init_params=ip_init, latent_dim=D,
        lengthscale=1.5, variance=1.0,
        sigma=sigma, sigma_drift_sq=sigma**2,
        eps_grid=1e-2, S_marginal=2,
        n_em_iters=n_em, n_estep_iters=n_estep, rho_sched=rho_sched,
        learn_emissions=True, update_R=False,
        learn_kernel=True, n_mstep_iters=n_mstep, mstep_lr=mstep_lr,
        n_hutchinson_mstep=16, kernel_warmup_iters=8,
        verbose=False,
    )
    wall = time.time() - t0
    return dict(mp=mp, op=op, hist=hist, wall=wall,
                ls=float(hist.lengthscale[-1]),
                var=float(hist.variance[-1]),
                ls_path=np.array(hist.lengthscale),
                var_path=np.array(hist.variance))

def run_sparsegp(lik, t_grid, op_init, ip_init, D, sigma, n_em, n_estep,
                 rho_sched, n_mstep, mstep_lr):
    quad = GaussHermiteQuadrature(D=D, n_quad=5)
    zs = initialize_zs(D=D, zs_lim=3.0, num_per_dim=8)   # 64 inducing pts
    sd = SparseGP(zs=zs, kernel=RBF(latent_dim=D), expectation=quad)
    sp0 = dict(length_scales=jnp.full((D,), 1.5), output_scale=jnp.asarray(1.0))
    dp_hist = []
    t0 = time.time()
    mp, _, gp_post, dp, _, op, _, elbos = fit_variational_em(
        key=jr.PRNGKey(33), fn=sd, likelihood=lik, t_grid=t_grid,
        drift_params=sp0, init_params=ip_init, output_params=op_init,
        sigma=sigma, rho_sched=rho_sched,
        n_iters=n_em, n_iters_e=n_estep, n_iters_m=n_mstep,
        perform_m_step=True, learn_output_params=True,
        learning_rate=mstep_lr * jnp.ones(n_em),
        print_interval=999, drift_params_history=dp_hist,
    )
    wall = time.time() - t0
    ls_path  = np.array([float(np.mean(h['length_scales'])) for h in dp_hist])
    var_path = np.array([float(h['output_scale'])**2 for h in dp_hist])
    return dict(mp=mp, op=op, sd=sd, gp_post=gp_post, dp=dp, wall=wall,
                ls=float(jnp.mean(dp['length_scales'])),
                var=float(dp['output_scale'])**2,
                ls_path=ls_path, var_path=var_path)

def eval_efgp_drift(efgp_res, t_grid, D, sigma, grid_pts_inferred):
    mp = efgp_res['mp']
    ls, var = efgp_res['ls'], efgp_res['var']
    m = jnp.asarray(mp['m'][0])
    S = jnp.asarray(np.asarray(mp['S'][0]))
    SS = jnp.asarray(np.asarray(mp['SS'][0]))
    del_t = t_grid[1:] - t_grid[:-1]
    X_tpl = jnp.linspace(-3., 3., 16)[:, None] * jnp.ones((1, D))
    grid = jp.spectral_grid_se(ls, var, X_tpl, eps=1e-2)
    mu_r, _, _ = jpd.compute_mu_r_jax(
        m, S, SS, del_t, grid, jr.PRNGKey(99),
        sigma_drift_sq=sigma**2, S_marginal=2, D_lat=D, D_out=D)
    Ef, _, _ = jpd.drift_moments_jax(
        mu_r, grid, jnp.asarray(grid_pts_inferred, dtype=jnp.float32),
        D_lat=D, D_out=D)
    return np.asarray(Ef)

def eval_sp_drift(sp_res, grid_pts_inferred):
    return np.asarray(sp_res['sd'].get_posterior_f_mean(
        sp_res['gp_post'], sp_res['dp'],
        jnp.asarray(grid_pts_inferred, dtype=jnp.float32)))

def _quiver(ax, GX, GY, F, color, title, xs_np=None):
    """Direction-only quiver (unit vectors) with optional trajectory overlay."""
    mag = np.linalg.norm(F, axis=1, keepdims=True).clip(1e-8)
    Fn = F / mag
    ax.quiver(GX, GY, Fn[:, 0].reshape(GX.shape), Fn[:, 1].reshape(GX.shape),
              color=color, angles='xy', scale_units='xy', scale=3.5, alpha=0.85,
              width=0.012, headwidth=3)
    if xs_np is not None:
        ax.plot(xs_np[:, 0], xs_np[:, 1], lw=0.6, color='red', alpha=0.3)
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal')
    ax.set_xlabel('$x_1$', fontsize=8); ax.set_ylabel('$x_2$', fontsize=8)
    ax.tick_params(labelsize=7)

In [ ]:
# NOTE: drift_fn must use jnp operations (JAX-traced by simulate_sde).
_A_rot = jnp.array([[-0.6, 1.0], [-1.0, -0.6]])

EXPS = [
    dict(
        name='Damped rotation',
        subtitle=r'$f(x)=Ax$, $A=[[-0.6,1],[-1,-0.6]]$ (linear)',
        drift_fn=lambda x, t: _A_rot @ x,
        x0=jnp.array([2.0, 0.0]),
        T=400, t_max=8.0, sigma=0.3, N_obs=8, seed=7,
    ),
    dict(
        name='Duffing double-well',
        subtitle=r'$f=[x_2,\; x_1-x_1^3-0.5x_2]$ (two stable foci at $x_1=\pm1$)',
        drift_fn=lambda x, t: jnp.stack([x[1], x[0] - x[0]**3 - 0.5*x[1]]),
        x0=jnp.array([1.2, 0.0]),
        T=400, t_max=15.0, sigma=0.2, N_obs=8, seed=13,
    ),
    dict(
        name='Anharmonic oscillator',
        subtitle=r'$f=[x_2,\; -x_1-0.3x_2-0.5x_1^3]$ (nonlinear single-well)',
        drift_fn=lambda x, t: jnp.stack([x[1], -x[0] - 0.3*x[1] - 0.5*x[0]**3]),
        x0=jnp.array([1.5, 0.0]),
        T=400, t_max=10.0, sigma=0.3, N_obs=8, seed=21,
    ),
]
D = 2
n_em    = 25
n_estep = 10
n_mstep = 10         # inner M-step Adam steps (same for both)
mstep_lr = 0.01      # Adam lr for kernel hypers (same for both)
rho_sched = jnp.logspace(-2, -1, n_em)   # 0.01→0.1, same for both
print(f'{len(EXPS)} experiments, n_em={n_em}, n_estep={n_estep}, \'
      f'n_mstep={n_mstep}, mstep_lr={mstep_lr}, \'
      f'rho {float(rho_sched[0]):.3f}→{float(rho_sched[-1]):.3f}')

In [ ]:
all_results = []

for cfg in EXPS:
    print(f'\n=== {cfg["name"]} ===')
    xs_np, ys = _make_data(cfg['drift_fn'], cfg['x0'], cfg['T'],
                            cfg['t_max'], cfg['sigma'], cfg['N_obs'], cfg['seed'])
    op_init, ip_init, t_grid = _shared_init(
        ys, D, cfg['N_obs'], cfg['t_max'], cfg['T'])

    # Eval grid (truth coords) ---
    lo = xs_np.min(0) - 0.4; hi = xs_np.max(0) + 0.4
    g0 = np.linspace(lo[0], hi[0], 14)
    g1 = np.linspace(lo[1], hi[1], 14)
    GX, GY = np.meshgrid(g0, g1, indexing='ij')
    grid_pts = np.stack([GX.ravel(), GY.ravel()], axis=-1).astype(np.float32)
    f_true = np.array([np.asarray(cfg['drift_fn'](
        jnp.asarray(p), 0.)) for p in grid_pts])
    zero_rmse = float(np.sqrt(np.mean(f_true**2)))

    # --- EFGP ---
    efgp_lik = _GLik(ys[None], jnp.ones((1, cfg['T']), dtype=bool))
    print('  EFGP...', end=' ', flush=True)
    efgp = run_efgp(efgp_lik, t_grid, op_init, ip_init, D,
                    cfg['sigma'], n_em, n_estep, rho_sched, n_mstep, mstep_lr)
    A_e, b_e = procrustes_align(efgp['mp']['m'][0], xs_np)
    lat_e = float(np.sqrt(np.mean(
        (np.asarray(efgp['mp']['m'][0]) @ A_e.T + b_e - xs_np)**2)))
    efgp['latent_rmse'] = lat_e
    # drift in truth basis
    grid_e = (grid_pts - b_e) @ np.linalg.inv(A_e).T
    f_efgp = eval_efgp_drift(efgp, t_grid, D, cfg['sigma'], grid_e) @ A_e.T
    efgp['drift_rmse'] = float(np.sqrt(np.mean((f_efgp - f_true)**2)))
    efgp['rel'] = efgp['drift_rmse'] / zero_rmse
    print(f'{efgp["wall"]:.1f}s  ell={efgp["ls"]:.3f}  '
          f'lat={lat_e:.4f}  drift={efgp["drift_rmse"]:.4f} ({efgp["rel"]:.1%})')

    # --- SparseGP ---
    sp_lik = _GLik(ys[None], jnp.ones((1, cfg['T']), dtype=bool))
    print('  SparseGP...', end=' ', flush=True)
    sp = run_sparsegp(sp_lik, t_grid, op_init, ip_init, D,
                      cfg['sigma'], n_em, n_estep, rho_sched, n_mstep, mstep_lr)
    A_s, b_s = procrustes_align(sp['mp']['m'][0], xs_np)
    lat_s = float(np.sqrt(np.mean(
        (np.asarray(sp['mp']['m'][0]) @ A_s.T + b_s - xs_np)**2)))
    sp['latent_rmse'] = lat_s
    # drift in truth basis
    grid_s = (grid_pts - b_s) @ np.linalg.inv(A_s).T
    f_sp = eval_sp_drift(sp, grid_s) @ A_s.T
    sp['drift_rmse'] = float(np.sqrt(np.mean((f_sp - f_true)**2)))
    sp['rel'] = sp['drift_rmse'] / zero_rmse
    print(f'{sp["wall"]:.1f}s  ell={sp["ls"]:.3f}  '
          f'lat={lat_s:.4f}  drift={sp["drift_rmse"]:.4f} ({sp["rel"]:.1%})')

    all_results.append(dict(
        cfg=cfg, xs_np=xs_np, t_grid=t_grid,
        GX=GX, GY=GY, grid_pts=grid_pts,
        f_true=f_true, f_efgp=f_efgp, f_sp=f_sp,
        zero_rmse=zero_rmse, efgp=efgp, sp=sp,
        A_e=A_e, b_e=b_e, A_s=A_s, b_s=b_s,
    ))

print('\nDone.')


## Per-experiment results

In [ ]:
for res in all_results:
    cfg  = res['cfg']
    efgp, sp = res['efgp'], res['sp']
    GX, GY   = res['GX'], res['GY']
    f_true, f_efgp, f_sp = res['f_true'], res['f_efgp'], res['f_sp']
    xs_np    = res['xs_np']
    n_iters_efgp = len(efgp['ls_path'])

    fig = plt.figure(figsize=(17, 7.5))
    gs  = mgs.GridSpec(2, 5, figure=fig, hspace=0.48, wspace=0.38,
                       width_ratios=[1, 1, 1, 1.15, 1.15])

    # ── Row 0: drift quivers ──────────────────────────────────────────────
    for col, (title, F, col_c) in enumerate([
        ('True drift (direction)', f_true, '0.35'),
        (f'EFGP  RMSE={efgp["drift_rmse"]:.3f}', f_efgp, 'C0'),
        (f'SparseGP  RMSE={sp["drift_rmse"]:.3f}',  f_sp,   'C2'),
    ]):
        ax = fig.add_subplot(gs[0, col])
        _quiver(ax, GX, GY, F, col_c, title, xs_np if col == 0 else None)

    # ── Row 1: hyperparameter paths ───────────────────────────────────────
    iters_e = np.arange(n_iters_efgp)
    iters_s = np.arange(len(sp['ls_path']))

    ax_ls = fig.add_subplot(gs[1, 0])
    ax_ls.plot(iters_e, efgp['ls_path'], color='C0', lw=1.8, label='EFGP')
    ax_ls.plot(iters_s, sp['ls_path'], color='C2', lw=1.8, ls='--', label='SparseGP')
    ax_ls.set_xlabel('EM iter', fontsize=8)
    ax_ls.set_ylabel('lengthscale \u2113', fontsize=8)
    ax_ls.set_title('\u2113 trajectory', fontsize=9)
    ax_ls.legend(fontsize=7, loc='best')
    ax_ls.tick_params(labelsize=7)

    ax_var = fig.add_subplot(gs[1, 1])
    ax_var.plot(iters_e, efgp['var_path'], color='C0', lw=1.8, label='EFGP')
    ax_var.plot(iters_s, sp['var_path'], color='C2', lw=1.8, ls='--', label='SparseGP')
    ax_var.set_xlabel('EM iter', fontsize=8)
    ax_var.set_ylabel(r'$\sigma^2$ (variance)', fontsize=8)
    ax_var.set_title(r'$\sigma^2$ trajectory', fontsize=9)
    ax_var.legend(fontsize=7, loc='best')
    ax_var.tick_params(labelsize=7)

    # ── Row 1 col 2: latent trajectories (truth vs inferred) ─────────────
    ax_lat = fig.add_subplot(gs[1, 2])
    m_e = np.asarray(efgp['mp']['m'][0]) @ res['A_e'].T + res['b_e']
    m_s = np.asarray(sp['mp']['m'][0])   @ res['A_s'].T + res['b_s']
    ax_lat.plot(xs_np[:, 0], xs_np[:, 1],
                color='gray', lw=1.0, alpha=0.7, label='truth')
    ax_lat.plot(m_e[:, 0], m_e[:, 1],
                color='C0', lw=0.9, alpha=0.7, label=f'EFGP ({efgp["latent_rmse"]:.3f})')
    ax_lat.plot(m_s[:, 0], m_s[:, 1],
                color='C2', lw=0.9, alpha=0.7, label=f'SparseGP ({sp["latent_rmse"]:.3f})')
    ax_lat.set_title('Latent recovery (Procrustes)', fontsize=9)
    ax_lat.set_xlabel('$x_1$', fontsize=8); ax_lat.set_ylabel('$x_2$', fontsize=8)
    ax_lat.legend(fontsize=6, loc='best'); ax_lat.tick_params(labelsize=7)
    ax_lat.set_aspect('equal')

    # ── Row 0+1 col 3-4: summary table ───────────────────────────────────
    ax_txt = fig.add_subplot(gs[:, 3:])
    ax_txt.axis('off')
    rows = [
        ('', 'EFGP', 'SparseGP'),
        ('Wall (s)', f'{efgp["wall"]:.1f}', f'{sp["wall"]:.1f}'),
        ('Latent RMSE', f'{efgp["latent_rmse"]:.4f}', f'{sp["latent_rmse"]:.4f}'),
        ('Drift RMSE', f'{efgp["drift_rmse"]:.4f}', f'{sp["drift_rmse"]:.4f}'),
        ('Rel. drift RMSE', f'{efgp["rel"]:.1%}', f'{sp["rel"]:.1%}'),
        ('Final \u2113', f'{efgp["ls"]:.3f}', f'{sp["ls"]:.3f}'),
        ('Final \u03c3\u00b2', f'{efgp["var"]:.3f}', f'{sp["var"]:.3f}'),
        ('', '', ''),
        ('zero-drift RMSE', f'{res["zero_rmse"]:.4f}', ''),
        ('T', str(cfg["T"]), f't_max={cfg["t_max"]}'),
        ('\u03c3_diff', str(cfg["sigma"]), ''),
    ]
    col_w = [0.38, 0.31, 0.31]
    y = 0.97
    for r in rows:
        x = 0.02
        for j, (txt, w) in enumerate(zip(r, col_w)):
            weight = 'bold' if (r[0] == '' and y > 0.92) else 'normal'
            if j == 0:
                weight = 'bold'
            ax_txt.text(x, y, txt, transform=ax_txt.transAxes,
                        fontsize=9, va='top', fontfamily='monospace',
                        fontweight=weight)
            x += w
        y -= 0.075

    fig.suptitle(f'{cfg["name"]}  \u2014  {cfg["subtitle"]}',
                 fontsize=11, fontweight='bold')
    plt.show()

## Summary: relative drift RMSE vs wall time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

exp_colors = ['C0', 'C3', 'C8']   # one color per experiment
from matplotlib.lines import Line2D

for i, res in enumerate(all_results):
    efgp, sp = res['efgp'], res['sp']
    name = res['cfg']['name']
    col  = exp_colors[i]

    xe, ye = efgp['wall'], efgp['rel']
    xs_, ys_ = sp['wall'],  sp['rel']

    ax.scatter(xe, ye, color=col, marker='o', s=110, zorder=5,
               edgecolors='k', linewidths=0.5)
    ax.scatter(xs_, ys_, color=col, marker='s', s=110, zorder=5,
               edgecolors='k', linewidths=0.5)
    ax.plot([xe, xs_], [ye, ys_], color=col, lw=1.2, ls='--', alpha=0.5)

    # label offset to avoid overlap
    ha_e = 'right' if xe > xs_ else 'left'
    ha_s = 'left'  if xe > xs_ else 'right'
    ax.annotate(f'EFGP\n{name}', (xe, ye),
                fontsize=7.5, ha=ha_e, va='bottom',
                xytext=(-6 if ha_e == 'right' else 6, 4),
                textcoords='offset points')
    ax.annotate(f'SparseGP\n{name}', (xs_, ys_),
                fontsize=7.5, ha=ha_s, va='top',
                xytext=(6 if ha_s == 'left' else -6, -4),
                textcoords='offset points')

legend_elems = [
    Line2D([0],[0], marker='o', color='gray', markersize=8, ls='', label='EFGP'),
    Line2D([0],[0], marker='s', color='gray', markersize=8, ls='', label='SparseGP'),
] + [Line2D([0],[0], color=exp_colors[i], lw=2.5,
            label=all_results[i]['cfg']['name'])
     for i in range(len(all_results))]
ax.legend(handles=legend_elems, fontsize=8, loc='upper left',
          framealpha=0.9)

ax.set_xlabel('Wall time (s)', fontsize=12)
ax.set_ylabel('Relative drift RMSE  (÷ zero-drift baseline)', fontsize=12)
ax.set_title('Speed vs accuracy  —  lower-left is better', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
